# 1D CNN for Network Intrusion Detection - Complete Implementation

**SOC Analyst Triage AI Assistant**

**⚠️ IMPORTANT:** This notebook is designed for Google Colab with GPU acceleration.

**Objective:** Implement 1D Convolutional Neural Network for automatic feature learning and compare with XGBoost baseline.

**Authors:** Napo Qheku & Kabelo Thesele  
**Supervisor:** Mr. Lekuba Ntoane  
**Date:** March 2026

---

## Why CNN?

**Advantages over Traditional ML (XGBoost):**
- **Automatic Feature Learning:** CNN learns features directly from raw data
- **Spatial Pattern Detection:** Captures local patterns in feature space
- **Hierarchical Representations:** Builds complex features from simple ones
- **No Manual Feature Engineering:** Reduces human bias

**Expected Outcomes:**
- May discover hidden attack patterns XGBoost missed
- Could match or exceed XGBoost performance
- Provides complementary strengths for ensemble

---

## Setup Instructions

### Step 1: Upload Data to Google Drive
Use the SAME preprocessed data from `02_preprocessing.ipynb`:
```
My Drive/SOC_Project/data/processed/
├── X_train.csv
├── X_test.csv
├── y_train.csv
└── y_test.csv
```

### Step 2: Enable GPU
Runtime → Change runtime type → T4 GPU → Save

### Step 3: Run All Cells
Runtime → Run all (estimated time: 30-45 minutes)

## 1. Mount Google Drive and Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")

In [ ]:
# Install TensorFlow if needed (usually pre-installed in Colab)
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Verify GPU
if tf.config.list_physical_devices('GPU'):
    print("\n✅ GPU is available and will be used for training!")
else:
    print("\n⚠️  GPU not detected. Training will be slower on CPU.")
    print("   Go to: Runtime → Change runtime type → Select GPU")

## 2. Import Libraries and Configuration

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import time
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import to_categorical

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, 
    roc_auc_score, roc_curve
)

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Paths
DRIVE_BASE = Path("/content/drive/MyDrive/SOC_Project")
PROCESSED_DIR = DRIVE_BASE / "data/processed"
MODEL_DIR = DRIVE_BASE / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"✅ Random seed: {RANDOM_SEED}")
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ NumPy version: {np.__version__}")
print(f"✅ Pandas version: {pd.__version__}")
print(f"\n📁 Paths:")
print(f"   Data: {PROCESSED_DIR}")
print(f"   Models: {MODEL_DIR}")
print("=" * 60)

## 3. Load Preprocessed Data (Same as XGBoost)

In [ ]:
print("=" * 60)
print("LOADING PREPROCESSED DATA")
print("=" * 60)

print("\n⏳ Loading data from Google Drive...\n")

# Load data
X_train_full = pd.read_csv(PROCESSED_DIR / "X_train.csv")
print("   ✓ X_train.csv loaded")

X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
print("   ✓ X_test.csv loaded")

y_train_full = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
print("   ✓ y_train.csv loaded")

y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()
print("   ✓ y_test.csv loaded")

print("\n✅ All data loaded successfully!")
print(f"\n📊 Dataset Shapes:")
print(f"   Training features: {X_train_full.shape}")
print(f"   Training labels:   {y_train_full.shape}")
print(f"   Test features:     {X_test.shape}")
print(f"   Test labels:       {y_test.shape}")
print(f"   Total features:    {X_train_full.shape[1]}")

# Verify data integrity
assert X_train_full.shape[0] == 175341, "Unexpected training set size"
assert X_test.shape[0] == 82332, "Unexpected test set size"
assert X_train_full.shape[1] == X_test.shape[1], "Feature dimension mismatch"

print("\n✅ Data integrity verified!")
print("=" * 60)

## 4. Train-Validation Split (Same as XGBoost for Fair Comparison)

In [ ]:
print("=" * 60)
print("TRAIN-VALIDATION SPLIT")
print("=" * 60)

# Same 80/20 split as XGBoost
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_SEED
)

print("\n📊 Dataset Splits:")
print(f"   Training:   {X_train.shape[0]:>7,} samples ({X_train.shape[0]/len(X_train_full)*100:>5.1f}%)")
print(f"   Validation: {X_val.shape[0]:>7,} samples ({X_val.shape[0]/len(X_train_full)*100:>5.1f}%)")
print(f"   Testing:    {X_test.shape[0]:>7,} samples (held-out)")
print(f"   Features:   {X_train.shape[1]:>7,}")

print("\n✓ Using SAME splits as XGBoost for fair comparison")
print("=" * 60)

## 5. Feature Scaling (Required for Neural Networks)

**Important:** Unlike tree-based models, neural networks require normalized features.

In [ ]:
print("=" * 60)
print("FEATURE SCALING")
print("=" * 60)

print("\n⚙️  Applying StandardScaler (zero mean, unit variance)...")

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Scaling complete!")
print(f"\n📊 Scaled Data Shapes:")
print(f"   Training:   {X_train_scaled.shape}")
print(f"   Validation: {X_val_scaled.shape}")
print(f"   Testing:    {X_test_scaled.shape}")

# Verify scaling
print(f"\n✓ Training mean: {X_train_scaled.mean():.6f} (should be ~0)")
print(f"✓ Training std:  {X_train_scaled.std():.6f} (should be ~1)")

# Reshape for CNN (add channel dimension)
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_val_cnn = X_val_scaled.reshape(X_val_scaled.shape[0], X_val_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print(f"\n📊 Reshaped for CNN (samples, features, channels):")
print(f"   Training:   {X_train_cnn.shape}")
print(f"   Validation: {X_val_cnn.shape}")
print(f"   Testing:    {X_test_cnn.shape}")

print("\n💡 Note: XGBoost doesn't need scaling, but CNNs do!")
print("=" * 60)

## 6. CNN Architecture Design

We'll test 3 architectures of increasing complexity.

In [ ]:
def create_simple_cnn(input_shape, name="SimpleCNN"):
    """
    Simple 1D CNN with 2 convolutional blocks.
    Good starting point for baseline.
    """
    model = models.Sequential(name=name)
    
    # Conv Block 1
    model.add(layers.Conv1D(64, kernel_size=3, activation='relu', 
                           padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.3))
    
    # Conv Block 2
    model.add(layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.3))
    
    # Global pooling
    model.add(layers.GlobalMaxPooling1D())
    
    # Dense layers
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model

def create_medium_cnn(input_shape, name="MediumCNN"):
    """
    Medium-depth 1D CNN with 3 convolutional blocks.
    More capacity for complex patterns.
    """
    model = models.Sequential(name=name)
    
    # Conv Block 1
    model.add(layers.Conv1D(64, kernel_size=5, activation='relu', 
                           padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(0.3))
    
    # Conv Block 2
    model.add(layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(0.3))
    
    # Conv Block 3
    model.add(layers.Conv1D(256, kernel_size=3, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.3))
    
    # Global pooling
    model.add(layers.GlobalMaxPooling1D())
    
    # Dense layers
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model

def create_deep_cnn(input_shape, name="DeepCNN"):
    """
    Deep 1D CNN with 4 convolutional blocks.
    Maximum capacity for automatic feature learning.
    """
    model = models.Sequential(name=name)
    
    # Conv Block 1
    model.add(layers.Conv1D(64, kernel_size=7, activation='relu', 
                           padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(0.2))
    
    # Conv Block 2
    model.add(layers.Conv1D(128, kernel_size=5, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(0.3))
    
    # Conv Block 3
    model.add(layers.Conv1D(256, kernel_size=3, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling1D(pool_size=2))
    model.add(layers.Dropout(0.3))
    
    # Conv Block 4
    model.add(layers.Conv1D(512, kernel_size=3, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.4))
    
    # Global pooling
    model.add(layers.GlobalMaxPooling1D())
    
    # Dense layers
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    return model

print("=" * 60)
print("CNN ARCHITECTURES DEFINED")
print("=" * 60)
print("\n✅ Three architectures ready:")
print("   1. SimpleCNN:  2 conv blocks (faster training)")
print("   2. MediumCNN:  3 conv blocks (balanced)")
print("   3. DeepCNN:    4 conv blocks (max capacity)")
print("\n💡 We'll train all three and compare performance")
print("=" * 60)

## 7. Training Configuration

In [ ]:
# Training hyperparameters
BATCH_SIZE = 256
EPOCHS = 50
LEARNING_RATE = 0.001

# Callbacks
def get_callbacks(model_name):
    return [
        # Early stopping
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        # Reduce learning rate on plateau
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        # Model checkpoint
        callbacks.ModelCheckpoint(
            filepath=str(MODEL_DIR / f"{model_name}_best.h5"),
            monitor='val_f1_score',
            mode='max',
            save_best_only=True,
            verbose=0
        )
    ]

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"\n📊 Hyperparameters:")
print(f"   Batch size:     {BATCH_SIZE}")
print(f"   Max epochs:     {EPOCHS}")
print(f"   Learning rate:  {LEARNING_RATE}")
print(f"   Optimizer:      Adam")
print(f"   Loss function:  Binary Crossentropy")
print(f"\n🔧 Callbacks:")
print(f"   • Early Stopping (patience=10)")
print(f"   • Learning Rate Reduction (patience=5)")
print(f"   • Model Checkpoint (save best)")
print("=" * 60)

## 8. Custom Metrics (F1-Score for Keras)

In [ ]:
# Custom F1-score metric for Keras
from tensorflow.keras import backend as K

def f1_score_keras(y_true, y_pred):
    """F1-score metric for binary classification"""
    y_pred = K.round(y_pred)
    tp = K.sum(K.cast(y_true * y_pred, 'float'), axis=0)
    fp = K.sum(K.cast((1 - y_true) * y_pred, 'float'), axis=0)
    fn = K.sum(K.cast(y_true * (1 - y_pred), 'float'), axis=0)
    
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())
    return f1

def recall_keras(y_true, y_pred):
    """Recall metric for binary classification"""
    y_pred = K.round(y_pred)
    tp = K.sum(K.cast(y_true * y_pred, 'float'), axis=0)
    fn = K.sum(K.cast(y_true * (1 - y_pred), 'float'), axis=0)
    return tp / (tp + fn + K.epsilon())

def precision_keras(y_true, y_pred):
    """Precision metric for binary classification"""
    y_pred = K.round(y_pred)
    tp = K.sum(K.cast(y_true * y_pred, 'float'), axis=0)
    fp = K.sum(K.cast((1 - y_true) * y_pred, 'float'), axis=0)
    return tp / (tp + fp + K.epsilon())

print("✅ Custom metrics defined for training monitoring")

## 9. Train SimpleCNN (Baseline)

Start with simplest architecture to establish baseline.

In [ ]:
print("\n" + "=" * 60)
print("TRAINING SIMPLE CNN")
print("=" * 60)

# Create model
input_shape = (X_train_cnn.shape[1], 1)  # (194, 1)
simple_cnn = create_simple_cnn(input_shape)

# Display architecture
print("\n📐 Model Architecture:")
simple_cnn.summary()

# Compile
simple_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', precision_keras, recall_keras, f1_score_keras]
)

print("\n⏳ Training SimpleCNN (this will take ~5-10 minutes)...\n")

# Train
start_time = time.time()

history_simple = simple_cnn.fit(
    X_train_cnn, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val_cnn, y_val),
    callbacks=get_callbacks("SimpleCNN"),
    verbose=1
)

training_time_simple = time.time() - start_time

print(f"\n✅ SimpleCNN training complete!")
print(f"   Training time: {training_time_simple/60:.1f} minutes")
print(f"   Epochs trained: {len(history_simple.history['loss'])}")
print("=" * 60)

## 10. Train MediumCNN

In [ ]:
print("\n" + "=" * 60)
print("TRAINING MEDIUM CNN")
print("=" * 60)

# Create model
medium_cnn = create_medium_cnn(input_shape)

# Display architecture
print("\n📐 Model Architecture:")
medium_cnn.summary()

# Compile
medium_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', precision_keras, recall_keras, f1_score_keras]
)

print("\n⏳ Training MediumCNN (this will take ~8-12 minutes)...\n")

# Train
start_time = time.time()

history_medium = medium_cnn.fit(
    X_train_cnn, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val_cnn, y_val),
    callbacks=get_callbacks("MediumCNN"),
    verbose=1
)

training_time_medium = time.time() - start_time

print(f"\n✅ MediumCNN training complete!")
print(f"   Training time: {training_time_medium/60:.1f} minutes")
print(f"   Epochs trained: {len(history_medium.history['loss'])}")
print("=" * 60)

## 11. Train DeepCNN

In [ ]:
print("\n" + "=" * 60)
print("TRAINING DEEP CNN")
print("=" * 60)

# Create model
deep_cnn = create_deep_cnn(input_shape)

# Display architecture
print("\n📐 Model Architecture:")
deep_cnn.summary()

# Compile
deep_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', precision_keras, recall_keras, f1_score_keras]
)

print("\n⏳ Training DeepCNN (this will take ~10-15 minutes)...\n")

# Train
start_time = time.time()

history_deep = deep_cnn.fit(
    X_train_cnn, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val_cnn, y_val),
    callbacks=get_callbacks("DeepCNN"),
    verbose=1
)

training_time_deep = time.time() - start_time

print(f"\n✅ DeepCNN training complete!")
print(f"   Training time: {training_time_deep/60:.1f} minutes")
print(f"   Epochs trained: {len(history_deep.history['loss'])}")
print("=" * 60)

## 12. Compare Training Histories (Learning Curves)

In [ ]:
print("=" * 60)
print("LEARNING CURVES COMPARISON")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

models_hist = [
    (history_simple, "SimpleCNN", "blue"),
    (history_medium, "MediumCNN", "green"),
    (history_deep, "DeepCNN", "red")
]

for idx, (hist, name, color) in enumerate(models_hist):
    # Loss
    axes[0, idx].plot(hist.history['loss'], label='Training', color=color, linewidth=2)
    axes[0, idx].plot(hist.history['val_loss'], label='Validation', 
                     color=color, linestyle='--', linewidth=2)
    axes[0, idx].set_title(f'{name}: Loss', fontweight='bold', fontsize=12)
    axes[0, idx].set_xlabel('Epoch')
    axes[0, idx].set_ylabel('Loss')
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)
    
    # F1-Score
    axes[1, idx].plot(hist.history['f1_score_keras'], label='Training', 
                     color=color, linewidth=2)
    axes[1, idx].plot(hist.history['val_f1_score_keras'], label='Validation', 
                     color=color, linestyle='--', linewidth=2)
    axes[1, idx].set_title(f'{name}: F1-Score', fontweight='bold', fontsize=12)
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('F1-Score')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'cnn_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Saved: cnn_learning_curves.png")
print("=" * 60)

## 13. Evaluate All Models on Test Set

In [ ]:
print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

models_to_eval = [
    (simple_cnn, "SimpleCNN"),
    (medium_cnn, "MediumCNN"),
    (deep_cnn, "DeepCNN")
]

results_comparison = []

for model, name in models_to_eval:
    print(f"\n{'='*60}")
    print(f"Evaluating {name}")
    print(f"{'='*60}")
    
    # Predictions
    y_pred_proba = model.predict(X_test_cnn, verbose=0).ravel()
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\n📊 Test Set Performance:")
    print(f"   Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
    print(f"   Precision: {prec:.4f} ({prec*100:.2f}%)")
    print(f"   Recall:    {rec:.4f} ({rec*100:.2f}%)")
    print(f"   F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
    print(f"   ROC-AUC:   {auc:.4f}")
    
    print(f"\n📊 Confusion Matrix:")
    print(f"   TN: {tn:>6,}  |  FP: {fp:>6,}")
    print(f"   FN: {fn:>6,}  |  TP: {tp:>6,}")
    print(f"\n   False Negative Rate: {fn/(fn+tp)*100:.2f}%")
    print(f"   False Positive Rate: {fp/(fp+tn)*100:.2f}%")
    
    # Store results
    results_comparison.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp)
    })

print("\n" + "=" * 60)
print("✅ All models evaluated!")
print("=" * 60)

## 14. Model Comparison Table

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(results_comparison)

print("\n" + "=" * 60)
print("CNN MODELS COMPARISON")
print("=" * 60)
print("\n")
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].to_string(index=False))
print("\n" + "=" * 60)

# Find best model
best_idx = results_df['F1-Score'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_f1 = results_df.loc[best_idx, 'F1-Score']

print(f"\n🏆 Best CNN Model: {best_model_name} (F1-Score: {best_f1:.4f})")
print("=" * 60)

# Save comparison
results_df.to_csv(MODEL_DIR / 'cnn_models_comparison.csv', index=False)
print("\n✓ Saved: cnn_models_comparison.csv")

## 15. Compare CNN vs XGBoost

Load XGBoost results and compare with best CNN.

In [ ]:
print("\n" + "=" * 60)
print("CNN vs XGBOOST COMPARISON")
print("=" * 60)

# Load XGBoost results
try:
    with open(MODEL_DIR / 'xgboost_results.json', 'r') as f:
        xgb_results = json.load(f)
    
    xgb_perf = xgb_results['test_performance']
    
    # Create comparison
    comparison_data = [
        {
            'Model': 'XGBoost (Baseline)',
            'Accuracy': xgb_perf['accuracy'],
            'Precision': xgb_perf['precision'],
            'Recall': xgb_perf['recall'],
            'F1-Score': xgb_perf['f1_score'],
            'ROC-AUC': xgb_perf['roc_auc']
        },
        {
            'Model': f'{best_model_name} (Best CNN)',
            'Accuracy': results_df.loc[best_idx, 'Accuracy'],
            'Precision': results_df.loc[best_idx, 'Precision'],
            'Recall': results_df.loc[best_idx, 'Recall'],
            'F1-Score': results_df.loc[best_idx, 'F1-Score'],
            'ROC-AUC': results_df.loc[best_idx, 'ROC-AUC']
        }
    ]
    
    final_comparison = pd.DataFrame(comparison_data)
    
    print("\n📊 Final Comparison:\n")
    print(final_comparison.to_string(index=False))
    
    # Calculate differences
    print("\n\n📈 Performance Differences (CNN vs XGBoost):\n")
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
        xgb_val = final_comparison.loc[0, metric]
        cnn_val = final_comparison.loc[1, metric]
        diff = cnn_val - xgb_val
        sign = "+" if diff > 0 else ""
        print(f"   {metric:12}: {sign}{diff:+.4f} ({sign}{diff*100:+.2f}%)")
    
    # Save final comparison
    final_comparison.to_csv(MODEL_DIR / 'xgboost_vs_cnn_comparison.csv', index=False)
    print("\n✓ Saved: xgboost_vs_cnn_comparison.csv")
    
except FileNotFoundError:
    print("\n⚠️  XGBoost results not found. Make sure to run 03_xgboost_optimized.ipynb first.")
    print("   Continuing with CNN results only...")

print("\n" + "=" * 60)

## 16. Visualization: Comparison Bar Chart

In [ ]:
if 'final_comparison' in locals():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    x = np.arange(len(metrics))
    width = 0.35
    
    xgb_values = final_comparison.loc[0, metrics].values
    cnn_values = final_comparison.loc[1, metrics].values
    
    bars1 = ax.bar(x - width/2, xgb_values, width, label='XGBoost', 
                   color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, cnn_values, width, label=best_model_name, 
                   color='coral', alpha=0.8)
    
    ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('XGBoost vs Best CNN: Performance Comparison', 
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylim([0.75, 1.0])
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'xgboost_vs_cnn_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Saved: xgboost_vs_cnn_comparison.png")

## 17. Save Best CNN Model and Results

In [ ]:
print("\n" + "=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Determine best model
best_models = {
    'SimpleCNN': simple_cnn,
    'MediumCNN': medium_cnn,
    'DeepCNN': deep_cnn
}
best_cnn_model = best_models[best_model_name]

# Save best model
best_cnn_model.save(MODEL_DIR / 'cnn_best_model.h5')
print(f"\n✓ Saved: cnn_best_model.h5 ({best_model_name})")

# Save comprehensive results
cnn_results = {
    'model_info': {
        'algorithm': 'CNN',
        'architecture': best_model_name,
        'tensorflow_version': tf.__version__,
        'trained_on': 'Google Colab',
        'random_seed': RANDOM_SEED
    },
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'epochs_trained': len(history_deep.history['loss']) if best_model_name == 'DeepCNN' else len(history_medium.history['loss']) if best_model_name == 'MediumCNN' else len(history_simple.history['loss']),
        'learning_rate': LEARNING_RATE,
        'optimizer': 'Adam'
    },
    'test_performance': {
        'accuracy': float(results_df.loc[best_idx, 'Accuracy']),
        'precision': float(results_df.loc[best_idx, 'Precision']),
        'recall': float(results_df.loc[best_idx, 'Recall']),
        'f1_score': float(results_df.loc[best_idx, 'F1-Score']),
        'roc_auc': float(results_df.loc[best_idx, 'ROC-AUC'])
    },
    'confusion_matrix': {
        'tn': int(results_df.loc[best_idx, 'TN']),
        'fp': int(results_df.loc[best_idx, 'FP']),
        'fn': int(results_df.loc[best_idx, 'FN']),
        'tp': int(results_df.loc[best_idx, 'TP'])
    },
    'all_architectures': results_comparison
}

with open(MODEL_DIR / 'cnn_results.json', 'w') as f:
    json.dump(cnn_results, f, indent=2)
print("✓ Saved: cnn_results.json")

# Save predictions
y_pred_best = best_cnn_model.predict(X_test_cnn, verbose=0).ravel()
predictions_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': (y_pred_best >= 0.5).astype(int),
    'y_proba': y_pred_best
})
predictions_df.to_csv(MODEL_DIR / 'cnn_predictions.csv', index=False)
print("✓ Saved: cnn_predictions.csv")

print("\n✅ All CNN results saved to Google Drive!")
print(f"\n📁 Location: {MODEL_DIR}")
print("   Files:")
print("   • cnn_best_model.h5")
print("   • cnn_results.json")
print("   • cnn_predictions.csv")
print("   • cnn_models_comparison.csv")
print("   • xgboost_vs_cnn_comparison.csv")
print("   • cnn_learning_curves.png")
print("   • xgboost_vs_cnn_comparison.png")
print("=" * 60)

## 18. Summary and Conclusions

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE!")
print("=" * 70)

print("\n✅ COMPLETED TASKS:")
tasks = [
    "Loaded preprocessed data (same as XGBoost)",
    "Applied feature scaling (required for CNNs)",
    "Designed 3 CNN architectures (Simple, Medium, Deep)",
    "Trained all models with proper callbacks",
    "Evaluated on held-out test set",
    "Compared CNN architectures",
    "Compared best CNN with XGBoost baseline",
    "Saved all models and results"
]
for i, task in enumerate(tasks, 1):
    print(f"   {i}. ✓ {task}")

if 'final_comparison' in locals():
    print("\n" + "=" * 70)
    print("FINAL RESULTS SUMMARY")
    print("=" * 70)
    
    print(f"\n🏆 Best CNN Model: {best_model_name}")
    print(f"\n📊 Performance:")
    print(f"   Accuracy:  {results_df.loc[best_idx, 'Accuracy']*100:>6.2f}%")
    print(f"   Precision: {results_df.loc[best_idx, 'Precision']*100:>6.2f}%")
    print(f"   Recall:    {results_df.loc[best_idx, 'Recall']*100:>6.2f}%")
    print(f"   F1-Score:  {results_df.loc[best_idx, 'F1-Score']*100:>6.2f}%")
    print(f"   ROC-AUC:   {results_df.loc[best_idx, 'ROC-AUC']:>6.4f}")
    
    print("\n📊 vs XGBoost Baseline:")
    xgb_f1 = xgb_perf['f1_score']
    cnn_f1 = results_df.loc[best_idx, 'F1-Score']
    diff = cnn_f1 - xgb_f1
    
    if diff > 0.005:  # More than 0.5% better
        print(f"   ✅ CNN outperforms XGBoost by {diff*100:.2f}% F1-score")
    elif diff < -0.005:  # More than 0.5% worse
        print(f"   ⚠️  XGBoost outperforms CNN by {abs(diff)*100:.2f}% F1-score")
    else:
        print(f"   ≈ CNN and XGBoost perform similarly (within 0.5%)")
    
    print("\n💡 Key Insights:")
    xgb_rec = xgb_perf['recall']
    cnn_rec = results_df.loc[best_idx, 'Recall']
    
    if cnn_rec > xgb_rec:
        print(f"   • CNN has better recall ({cnn_rec*100:.2f}% vs {xgb_rec*100:.2f}%)")
    else:
        print(f"   • XGBoost has better recall ({xgb_rec*100:.2f}% vs {cnn_rec*100:.2f}%)")
    
    print("   • CNN learned features automatically (no manual engineering)")
    print("   • Both models provide complementary strengths for ensemble")

print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)
print("   1. Implement LSTM for temporal sequence modeling")
print("   2. Create ensemble combining XGBoost + CNN + LSTM")
print("   3. Integrate models into dashboard")
print("   4. Conduct user testing with SOC analysts")

print("\n" + "=" * 70)
print("✨ CNN IMPLEMENTATION COMPLETE! ✨")
print("=" * 70 + "\n")